# Reconocimiento de dígitos manuscritos con CNN

## Proyecto de portafolio — Deep Learning / Computer Vision

**Objetivo:** construir, evaluar y optimizar una red neuronal convolutiva para clasificar dígitos manuscritos del 0 al 9 a partir de imágenes de 8×8 píxeles en escala de grises.

**Contexto de negocio:** este tipo de solución puede integrarse en sistemas de digitalización de formularios, lectura automática de códigos y automatización documental.

**Tecnologías:** Python · Pandas · NumPy · Matplotlib · Scikit-learn · TensorFlow · Keras


## 1. ¿Por qué usar una CNN?

Las imágenes contienen relaciones espaciales entre píxeles. Una CNN aprovecha esa estructura mediante filtros que detectan patrones locales.

Flujo general:

\[
\text{Imagen} \rightarrow \text{Conv2D} \rightarrow \text{ReLU} \rightarrow \text{Pooling} \rightarrow \text{Flatten} \rightarrow \text{Dense} \rightarrow \text{Softmax}
\]

Las primeras capas pueden aprender bordes y trazos; capas posteriores combinan esas características para distinguir los dígitos.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis("off")
bloques = [
    ("Input","8×8×1"),("Conv2D","32 filtros"),("Pooling","2×2"),
    ("Conv2D","64 filtros"),("Flatten","Vector"),("Dense","128"),
    ("Output","10 + Softmax")
]
xs = [0.05,0.2,0.35,0.5,0.65,0.8,0.95]
for i, ((a,b), x) in enumerate(zip(bloques,xs)):
    ax.text(x,0.5,f"{a}\n{b}",ha="center",va="center",
            bbox=dict(boxstyle="round,pad=0.4"))
    if i < len(xs)-1:
        ax.annotate("", xy=(xs[i+1]-0.05,0.5), xytext=(x+0.05,0.5),
                    arrowprops=dict(arrowstyle="->"))
ax.set_title("Arquitectura conceptual de la CNN")
plt.show()


## 2. Importación de librerías

El notebook está preparado para Jupyter o Google Colab. En un entorno local puede ser necesario instalar `tensorflow` y `openpyxl`.


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## 3. Carga del dataset

Estructura recomendada del repositorio:

```text
cnn-reconocimiento-digitos/
├── data/
│   └── digitos_mnist_simple.xlsx
├── notebooks/
│   └── reconocimiento_digitos_cnn.ipynb
└── README.md
```


In [ ]:
rutas = [
    "../data/digitos_mnist_simple.xlsx",
    "data/digitos_mnist_simple.xlsx",
    "digitos_mnist_simple.xlsx",
    "07. Apoyo desafío - digitos_mnist_simple.xlsx",
    "/content/digitos_mnist_simple.xlsx",
    "/content/07. Apoyo desafío - digitos_mnist_simple.xlsx",
    "/mnt/data/07. Apoyo desafío - digitos_mnist_simple.xlsx"
]

DATA_PATH = next((p for p in rutas if os.path.exists(p)), None)

if DATA_PATH is None:
    raise FileNotFoundError("No se encontró el dataset. Ajusta DATA_PATH o ubícalo en data/.")

df = pd.read_excel(DATA_PATH)
print("Ruta:", DATA_PATH)
print("Dimensiones:", df.shape)
display(df.head())


## 4. Análisis exploratorio

El dataset contiene 1.797 observaciones, 64 variables de píxeles y una variable objetivo `label`.


In [ ]:
print("Nulos:", df.isna().sum().sum())
print("Clases:", sorted(df["label"].unique()))

X_raw = df.drop(columns="label").values.astype("float32")
y = df["label"].values.astype("int64")

print("Forma X:", X_raw.shape)
print("Forma y:", y.shape)
print("Píxel mínimo:", X_raw.min())
print("Píxel máximo:", X_raw.max())

print("\nDistribución:")
print(df["label"].value_counts().sort_index())


In [ ]:
conteo = df["label"].value_counts().sort_index()

plt.figure(figsize=(8,4))
conteo.plot(kind="bar")
plt.title("Distribución de clases")
plt.xlabel("Dígito")
plt.ylabel("Cantidad")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()


### Hallazgo

Las diez clases están bien balanceadas, por lo que `accuracy` es una métrica principal útil. Aun así, el proyecto también incorpora matriz de confusión, precision, recall y F1-score.


## 5. Visualización de imágenes

Cada fila representa 64 píxeles:

\[
8\times8 = 64
\]


In [ ]:
fig, axes = plt.subplots(2,5,figsize=(10,5))

for digito in range(10):
    idx = np.where(y == digito)[0][0]
    img = X_raw[idx].reshape(8,8)
    ax = axes[digito//5, digito%5]
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Dígito {digito}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 6. Preprocesamiento

Los píxeles se encuentran entre 0 y 16, por lo que se normalizan mediante:

\[
X_{norm}=\frac{X}{16}
\]

Luego se transforma la matriz de `(n,64)` a `(n,8,8,1)` para que la CNN reciba imágenes con un canal.


In [ ]:
X = X_raw / 16.0
X = X.reshape(-1, 8, 8, 1)

print("Forma CNN:", X.shape)
print("Rango:", X.min(), X.max())


## 7. División Train / Validation / Test

- 60% entrenamiento
- 20% validación
- 20% prueba

Se utiliza `stratify` para conservar la proporción de cada dígito.


In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,
    random_state=SEED, stratify=y_train_full
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# 8. Modelo base

La arquitectura base funciona como referencia para medir el impacto de las optimizaciones posteriores.


In [ ]:
def crear_modelo_base():
    modelo = Sequential([
        Input(shape=(8,8,1)),
        Conv2D(32, (3,3), activation="relu", padding="same"),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(64, activation="relu"),
        Dense(10, activation="softmax")
    ])
    modelo.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

modelo_base = crear_modelo_base()
modelo_base.summary()


In [ ]:
history_base = modelo_base.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    verbose=1
)


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history_base.history["loss"], label="Train")
plt.plot(history_base.history["val_loss"], label="Validation")
plt.title("Modelo base - Loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history_base.history["accuracy"], label="Train")
plt.plot(history_base.history["val_accuracy"], label="Validation")
plt.title("Modelo base - Accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
base_loss, base_acc = modelo_base.evaluate(X_test, y_test, verbose=0)
y_prob_base = modelo_base.predict(X_test, verbose=0)
y_pred_base = np.argmax(y_prob_base, axis=1)

print(f"Accuracy base: {base_acc*100:.2f}%")
print(f"Loss base: {base_loss:.4f}")
print("\n", classification_report(y_test, y_pred_base, digits=4))


In [ ]:
cm_base = confusion_matrix(y_test, y_pred_base)
ConfusionMatrixDisplay(cm_base, display_labels=list(range(10))).plot(values_format="d")
plt.title("Matriz de confusión - Modelo base")
plt.show()


# 9. Modelo optimizado

Se aplican tres mejoras:

1. Segundo bloque convolutivo.
2. `Dropout(0.30)` para reducir sobreajuste.
3. `EarlyStopping` para detener el entrenamiento cuando `val_loss` deja de mejorar.

También se explicita `learning_rate=0.001`.


In [ ]:
def crear_modelo_optimizado():
    modelo = Sequential([
        Input(shape=(8,8,1)),
        Conv2D(32, (3,3), activation="relu", padding="same"),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation="relu", padding="same"),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.30),
        Dense(10, activation="softmax")
    ])
    modelo.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

modelo_opt = crear_modelo_optimizado()
modelo_opt.summary()


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history_opt = modelo_opt.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

print("Épocas ejecutadas:", len(history_opt.history["loss"]))


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history_opt.history["loss"], label="Train")
plt.plot(history_opt.history["val_loss"], label="Validation")
plt.title("Modelo optimizado - Loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history_opt.history["accuracy"], label="Train")
plt.plot(history_opt.history["val_accuracy"], label="Validation")
plt.title("Modelo optimizado - Accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
opt_loss, opt_acc = modelo_opt.evaluate(X_test, y_test, verbose=0)
y_prob_opt = modelo_opt.predict(X_test, verbose=0)
y_pred_opt = np.argmax(y_prob_opt, axis=1)

print(f"Accuracy optimizado: {opt_acc*100:.2f}%")
print(f"Loss optimizado: {opt_loss:.4f}")
print("\n", classification_report(y_test, y_pred_opt, digits=4))


In [ ]:
cm_opt = confusion_matrix(y_test, y_pred_opt)
ConfusionMatrixDisplay(cm_opt, display_labels=list(range(10))).plot(values_format="d")
plt.title("Matriz de confusión - Modelo optimizado")
plt.show()


# 10. Comparación de modelos

La comparación se realiza sobre el conjunto de prueba, que no intervino en el ajuste de pesos ni en la selección de hiperparámetros.


In [ ]:
comparacion = pd.DataFrame({
    "Modelo": ["Base", "Optimizado"],
    "Accuracy": [base_acc, opt_acc],
    "Loss": [base_loss, opt_loss]
})

comparacion["Accuracy (%)"] = comparacion["Accuracy"] * 100
display(comparacion.round(4))

print(f"Cambio en accuracy: {(opt_acc-base_acc)*100:+.2f} puntos porcentuales")
print(f"Cambio en loss: {opt_loss-base_loss:+.4f}")


## Reflexión técnica

La optimización no consiste en agregar capas sin justificación.

Un modelo mejor debe demostrar mayor capacidad de generalización o mayor estabilidad en validación. Si el modelo optimizado mejora el accuracy o reduce la pérdida en test, existe evidencia de una mejora real. Si las métricas son similares pero las curvas de validación son más estables, la regularización también aporta valor. Si empeora, el resultado indica que la complejidad adicional no fue necesaria para este dataset.


# 11. Análisis visual de predicciones y errores


In [ ]:
fig, axes = plt.subplots(3,5,figsize=(10,7))

for ax, idx in zip(axes.ravel(), range(15)):
    ax.imshow(X_test[idx].reshape(8,8), cmap="gray")
    ax.set_title(f"Real {y_test[idx]} | Pred {y_pred_opt[idx]}")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
errores = np.where(y_pred_opt != y_test)[0]
print("Total de errores:", len(errores))

if len(errores) > 0:
    n = min(10, len(errores))
    fig, axes = plt.subplots(2,5,figsize=(10,5))
    axes = axes.ravel()
    for ax in axes:
        ax.axis("off")
    for ax, idx in zip(axes, errores[:n]):
        ax.imshow(X_test[idx].reshape(8,8), cmap="gray")
        ax.set_title(f"Real {y_test[idx]} / Pred {y_pred_opt[idx]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# 12. Función de inferencia


In [ ]:
def predecir_digito(imagen_8x8, modelo=modelo_opt):
    imagen = np.asarray(imagen_8x8, dtype="float32")
    if imagen.shape != (8,8):
        raise ValueError("La imagen debe tener forma 8x8.")
    if imagen.max() > 1:
        imagen = imagen / 16.0
    imagen = imagen.reshape(1,8,8,1)
    probs = modelo.predict(imagen, verbose=0)[0]
    pred = int(np.argmax(probs))
    return pred, probs

pred, probs = predecir_digito(X_test[0].reshape(8,8))
print("Predicción:", pred)
print("Clase real:", y_test[0])
print("Confianza:", probs[pred])


# 13. Guardado del modelo


In [ ]:
modelo_opt.save("cnn_digitos.keras")
print("Modelo guardado como cnn_digitos.keras")


# 14. Conclusiones

Este proyecto cubre un flujo completo de Computer Vision con Deep Learning:

- exploración del dataset;
- visualización de imágenes;
- normalización;
- reshape a tensor 4D;
- división train/validation/test;
- construcción de una CNN base;
- evaluación con múltiples métricas;
- regularización mediante Dropout;
- EarlyStopping;
- comparación objetiva de modelos;
- análisis de errores;
- función de inferencia;
- guardado del modelo.

La principal conclusión es que las CNN son adecuadas para imágenes porque explotan la estructura espacial de los píxeles. Sin embargo, la calidad de un modelo debe evaluarse sobre datos no vistos y no solo a partir del rendimiento de entrenamiento.


# 15. Próximas mejoras

- Data augmentation.
- Batch Normalization.
- Ajuste sistemático de hiperparámetros.
- Comparación con una red densa.
- Validación cruzada.
- API de inferencia.
- Interfaz web para dibujar un dígito y clasificarlo en tiempo real.

---

## Autor

**Héctor A. López Giménez**  
Portafolio de Data Science · Machine Learning · Deep Learning
